In [46]:
#!pip install seaborn
#!pip install hf_xet

In [71]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt

import sklearn.metrics as metrics
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

## Modelos nulos para comparar

In [72]:
dataset = load_dataset("mirluvams/popocatepetl")
test_dataset = dataset["test"]

y_test = test_dataset["label"]
print("Número de muestras:", len(y_test))
print("Ejemplo de etiquetas:", y_test[:10])

Número de muestras: 24040
Ejemplo de etiquetas: [3, 2, 3, 0, 0, 3, 0, 0, 3, 3]


In [73]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd

# Datos ficticios para cumplir con la forma de X
X_test = np.zeros((len(y_test), 1))  
y_test_array = np.array(y_test)

strategies = {
    "most_frequent": "Clase mayoritaria",
    "stratified": "Aleatorio ponderado (según distribución)",
    "uniform": "Aleatorio uniforme"
}

results = []

for strategy, name in strategies.items():
    dummy = DummyClassifier(strategy=strategy, random_state=42)
    dummy.fit(X_test, y_test_array)
    y_pred = dummy.predict(X_test)
    
    acc = accuracy_score(y_test_array, y_pred)
    f1_weighted = f1_score(y_test_array, y_pred, average='weighted')  # F1 ponderado (igual que en tus modelos)
    f1_micro = f1_score(y_test_array, y_pred, average='micro')
    f1_macro = f1_score(y_test_array, y_pred, average='macro')
    
    results.append({
        "model": name,
        "accuracy": acc,
        "f1": f1_weighted,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro
    })

df_baselines = pd.DataFrame(results)
print("\nResultados de modelos nulos:\n")
display(df_baselines)


Resultados de modelos nulos:



,model,accuracy,f1,f1_micro,f1_macro
0,Clase mayoritaria,0.384318,0.213391,0.384318,0.138811
1,Aleatorio ponderado (según distribución),0.324709,0.324376,0.324709,0.252881
2,Aleatorio uniforme,0.252080,0.272777,0.252080,0.230795


In [77]:
# cargar resultados de modelos finetuneados
df_results_finetuned = pd.read_csv("results_finetuned.csv")

In [78]:
# solo columnas clave
df_results_finetuned = df_results_finetuned[["model", "accuracy", "f1", "f1_micro", "f1_macro"]]

In [79]:
# simplificar nombres
df_results_finetuned["model"] = df_results_finetuned["model"].apply(
    lambda x: x.split("/")[-1].replace("-popocatepetl", "")
)

In [80]:
# combinar con los modelos nulos
df_comparacion = pd.concat([df_baselines, df_results_finetuned], ignore_index=True)

In [81]:
# agregar columna para distinguir baselines y transformers finetuneados
df_comparacion["tipo"] = df_comparacion["model"].apply(
    lambda x: "Nulo" if x in df_baselines["model"].values else "Finetuneado"
)

In [82]:
# Ordenar por F1-macro descendente
df_comparacion = df_comparacion.sort_values(by="f1_macro", ascending=False).reset_index(drop=True)

In [83]:
# mostrar tabla final
df_comparacion

,model,accuracy,f1,f1_micro,f1_macro,tipo
0,swinv2-base-patch4-window16-256,0.989434,0.989440,0.989434,0.986394,Finetuneado
1,vit-base-patch16-224,0.987438,0.987449,0.987438,0.983497,Finetuneado
2,convnext-base-224,0.986897,0.986932,0.986897,0.983164,Finetuneado
3,beit-base-patch16-224,0.986273,0.986286,0.986273,0.982046,Finetuneado
4,dinov2-base,0.978577,0.978651,0.978577,0.972463,Finetuneado
5,Aleatorio ponderado (según distribución),0.324709,0.324376,0.324709,0.252881,Nulo
6,Aleatorio uniforme,0.252080,0.272777,0.252080,0.230795,Nulo
7,Clase mayoritaria,0.384318,0.213391,0.384318,0.138811,Nulo


In [84]:
# guardar csv
df_comparacion.to_csv("comparacion_modelos.csv")